In [28]:
#imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [29]:
df = pd.read_csv("/home/desktop/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/clean_undersampled.csv", low_memory=False)

In [30]:
#MACHINE LEARNIGN TIME!!
y = df['profitable']
X = df.drop(columns=['profitable'])

In [31]:
# spliting the test set and the training set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

# splitting the training portion into train and validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1765, random_state = 42, stratify=y_train)

In [32]:
#Now we have an 80/15/15 split with our data
print(len(X_train), len(X_val), len(X_test))
print(y_train.mean(), y_val.mean(), y_test.mean())  # all should be ~0.5

384407 82390 82377
0.5000013007047218 0.5 0.499993930344635


In [33]:
class Model(nn.Module):
    #Input Layer, 150 Features --> Hidden Layer 1 --> Hidden Layer 2 --> ouput (profitable or not)
    def __init__(self, in_features = 151, h1 = 130, h2 =120, out_features = 1):
        super().__init__()
        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)

        return x
    

In [34]:
# p[icka manual seed for randomization
torch.manual_seed(42)

In [35]:
model = Model()

In [36]:
#preprocessor pipleine. Filling in Nans and scaling
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
# Fit on train, transform all
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(X_train_proc.shape)  # should be (384407, 151)

(384407, 151)


In [37]:
# Must convert the data into tensors. Neuralnetwork only work with tensors. Multi
X_train_t = torch.tensor(X_train_proc, dtype=torch.float32)
X_val_t = torch.tensor(X_val_proc, dtype=torch.float32)
X_test_t = torch.tensor(X_test_proc, dtype=torch.float32)

In [38]:
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32)

In [39]:
# LAoding into a data loader, a  data laoder lads our data then hands it to us in chunks called batches. 
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

val_ds = TensorDataset(X_val_t, y_val_t)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)

In [40]:
#loss optimizer
criterion = nn.BCEWithLogitsLoss() #binary corss entropy with built in sigmoid
#sigmoid: squashes any number between 0 and 1
#cross entropy measure hopw different the models prediction and actaul value. 

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Training loop
epochs = 10
train_losses = []
val_losses = []

for epoch in range(1, epochs + 1):
    # --- Training phase ---
    model.train()
    running_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()          # clear gradients from previous batch
        
        logits = model(X_batch)        # shape: (512, 1)
        logits = logits.squeeze()      # shape: (512,) to match y_batch
        
        loss = criterion(logits, y_batch)
        loss.backward()                # compute gradients
        optimizer.step()               # update weights
        
        running_loss += loss.item()    # accumulate (item() converts to plain float)
    
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():              # don't track gradients for val (saves memory)
        for X_batch, y_batch in val_loader:
            logits = model(X_batch).squeeze()
            loss = criterion(logits, y_batch)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    print(f"Epoch {epoch:2d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

Epoch  1 | Train Loss: 0.6092 | Val Loss: 0.5999
